<a href="https://colab.research.google.com/github/edolio/ML_Projects/blob/main/LogisticRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

# data are supposed to be in personal drive
drive.mount('/content/drive')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold, GridSearchCV, train_test_split, cross_validate
from sklearn import metrics
from sklearn.compose import ColumnTransformer
from sklearn.metrics import log_loss, ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import seaborn as sns
from pathlib import Path
from PIL import Image
import numpy as np
from matplotlib import pyplot as plt

label_cloudy = 1
label_desert = 2
label_green = 3
label_water = 4
class_names = ['cloudy', 'desert', 'green', 'water']   # must match the order of logisticRegr.classes_

**DATASET PREPARATION and MODEL INITIALIZATION**

In [ ]:
# Image dimension is limited to 64x64 for RAM constraints
folder = Path('/content/drive/MyDrive/Training data/data/cloudy')  # change to your folder
paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'})
images_cloudy = [np.array(Image.open(p).convert('RGB').resize((64, 64))) for p in paths]
Y = np.full(len(images_cloudy), label_cloudy)

folder = Path('/content/drive/MyDrive/Training data/data/desert')  # change to your folder
paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'})
images_desert = [np.array(Image.open(p).convert('RGB').resize((64, 64))) for p in paths]
Y = np.concatenate([Y, np.full(len(images_desert), label_desert)])

folder = Path('/content/drive/MyDrive/Training data/data/green_area')  # change to your folder
paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'})
images_green = [np.array(Image.open(p).convert('RGB').resize((64, 64))) for p in paths]
Y = np.concatenate([Y, np.full(len(images_green), label_green)])

folder = Path('/content/drive/MyDrive/Training data/data/water')  # change to your folder
paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'})
images_water = [np.array(Image.open(p).convert('RGB').resize((64, 64))) for p in paths]
Y = np.concatenate([Y, np.full(len(images_water), label_water)])

print("Image shape is: " + str(images_cloudy[0].shape))
print("Print example image\n")
plt.imshow(images_cloudy[0], interpolation='nearest')
plt.show()

# data is normalized
X = np.stack(images_cloudy + images_desert + images_green + images_water,
             axis=0).reshape(len(Y), -1).astype('float32')

# delete unused arrays otherwise Colab collapses beacuse of lack of RAM
del images_cloudy, images_desert, images_green, images_water

print("Input shape is: " + str(X.shape))
print("Output shape is: " + str(Y.shape))

# model object initialization
# max_iter = 500 just for computation speed
logisticRegr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))

# useful function for later
def print_results(model, x_train, y_train, x_test, y_test, class_names):
    score = model.score(x_test, y_test)
    print(score)

    # plot confusion Matrix ----------------------------------------------
    y_pred = model.predict(x_test)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=class_names,
        normalize='true', values_format='.1%',
        cmap='Blues', colorbar=False, ax=ax)

    ax.set_title('Confusion matrix (test set)', fontsize=14, pad=12)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    plt.tight_layout()
    plt.show()
    # plot confusion Matrix ----------------------------------------------

    train_loss = log_loss(y_train, model.predict_proba(x_train), labels=model.classes_)
    test_loss = log_loss(y_test, model.predict_proba(x_test), labels=model.classes_)

    print(f'train log loss: {train_loss:.4f}')
    print(f'test  log loss: {test_loss:.4f}')
    print(f'train acc: {model.score(x_train, y_train):.4f}   test acc: {model.score(x_test, y_test):.4f}')


**VALIDATION STEP - PERFORM STRATIFIED K-FOLD VALIDATION**



In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True)


grid = GridSearchCV(
    logisticRegr,
    {'logisticregression__C': [0.01, 0.1, 1]},
    cv=skf,
    scoring={'acc': 'accuracy', 'll': 'neg_log_loss'},
    refit='acc',                 # best_estimator_ is chosen by accuracy
    return_train_score=True)
grid.fit(X, Y)

r = grid.cv_results_
for i, params in enumerate(r['params']):
    print(f"{params}  "
          f"acc train {r['mean_train_acc'][i]:.4f}  test {r['mean_test_acc'][i]:.4f} ± {r['std_test_acc'][i]:.4f}  |  "
          f"log loss train {-r['mean_train_ll'][i]:.4f}  test {-r['mean_test_ll'][i]:.4f} ± {r['std_test_ll'][i]:.4f}")

print('best C:', grid.best_params_, f'acc {grid.best_score_:.4f}')

**FINAL MODEL TRAIN AND TEST**

In [ ]:
# change C according to validation
logisticRegr = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=500))

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.15, random_state=0)
logisticRegr.fit(x_train, y_train)

print_results(logisticRegr, x_train, y_train, x_test, y_test, class_names)

**VECTORIAL PCA IN PIPELINE - Repeat Validation**

In [ ]:
logisticRegr = make_pipeline(StandardScaler(), PCA(n_components=0.95), LogisticRegression(max_iter=500))

skf = StratifiedKFold(n_splits=3, shuffle=True)


grid = GridSearchCV(
    logisticRegr,
    {'logisticregression__C': [0.001, 0.1, 100]},
    cv=skf,
    scoring={'acc': 'accuracy', 'll': 'neg_log_loss'},
    refit='acc',                 # best_estimator_ is chosen by accuracy
    return_train_score=True)
grid.fit(X, Y)

r = grid.cv_results_
for i, params in enumerate(r['params']):
    print(f"{params}  "
          f"acc train {r['mean_train_acc'][i]:.4f}  test {r['mean_test_acc'][i]:.4f} ± {r['std_test_acc'][i]:.4f}  |  "
          f"log loss train {-r['mean_train_ll'][i]:.4f}  test {-r['mean_test_ll'][i]:.4f} ± {r['std_test_ll'][i]:.4f}")

print('best C:', grid.best_params_, f'acc {grid.best_score_:.4f}')

**Train Final Model with PCA**

In [ ]:
logisticRegr = make_pipeline(StandardScaler(), PCA(n_components=0.95), LogisticRegression(C=0.1,max_iter=500))

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.15, random_state=0, stratify=Y)
logisticRegr.fit(x_train, y_train)

print_results(logisticRegr, x_train, y_train, x_test, y_test, class_names)

**TEXTURE FEAUTURE** - We will add a feature based on texture difference between green and water images

In [ ]:
def texture_features(X_flat, size=64):
    g = X_flat.reshape(-1, size, size, 3).astype('float32').mean(axis=3)  # greyscale, shape (N, size, size)
    dx = np.abs(np.diff(g, axis=2))   # difference between each pixel and its right neighbour
    dy = np.abs(np.diff(g, axis=1))   # difference between each pixel and the one below
    return np.column_stack([
        dx.mean(axis=(1, 2)), dy.mean(axis=(1, 2)),   # average graininess
        dx.std(axis=(1, 2)),  dy.std(axis=(1, 2)),    # is the graininess uniform or patchy?
        g.std(axis=(1, 2)),                           # overall contrast
    ])

T = texture_features(X)   # shape (5631, 5)

# if class 3 and 4 show different numbers, texture feature will be effective
for c in np.unique(Y):
    print(c, T[Y == c].mean(axis=0).round(1))

In [ ]:
n_pix = X.shape[1]
X_all = np.hstack([X.astype('float32'), T])

pre = ColumnTransformer([
    ('pixels',  make_pipeline(StandardScaler(), PCA(n_components=0.95)), slice(0, n_pix)),
    ('texture', StandardScaler(), slice(n_pix, None)),
])

logisticRegr = make_pipeline(pre, LogisticRegression(C=0.1, max_iter=500))

x_train, x_test, y_train, y_test = train_test_split(X_all, Y, test_size=0.15, random_state=0, stratify=Y)
logisticRegr.fit(x_train, y_train)

print_results(logisticRegr, x_train, y_train, x_test, y_test, class_names)

**SAVE THE MODEL TO DRIVE**

In [ ]:
import joblib, os

save_dir = '/content/drive/MyDrive/TrainedModels'        # change to the folder you want
os.makedirs(save_dir, exist_ok=True)
save_path = f'{save_dir}/LogisticRegr_landclass.joblib'

joblib.dump(logisticRegr, save_path)
print('saved to', save_path)

**LOAD MODEL and GENERATE UI**

In [ ]:
import gradio as gr

model = joblib.load('/content/drive/MyDrive/TrainedModels/LogisticRegr_landclass.joblib')
print(model.classes_)

SIZE = 64   # must match the size you trained with

def preprocess(img):
    x = np.array(img.convert('RGB').resize((SIZE, SIZE))).reshape(1, -1).astype('float32')
    return np.hstack([x, texture_features(x, SIZE)])   # same columns as X_all; use just `return x` if you trained on pixels only

def predict(img):
    probs = model.predict_proba(preprocess(img))[0]
    label = {str(c): float(p) for c, p in zip(model.classes_, probs)}
    table = [[str(c), f'{p:.1%}'] for c, p in sorted(zip(model.classes_, probs), key=lambda t: -t[1])]
    return label, table

gr.Interface(
    fn=predict,
    inputs=gr.Image(type='pil', label='Upload an image'),
    outputs=[
        gr.Label(num_top_classes=4, label='Predicted class'),
        gr.Dataframe(headers=['class', 'probability'], label='All probabilities'),
    ],
    title='Land cover classifier',
).launch()